
One of the most interesting applications of JSON outputs in agentic development is *routing* the workflow to different actions depending on the LLM's assessment.

Prompt a model with a collection of context and criteria and then ask it to create a decision. That decision can then be parsed as JSON and used to shape complex, versatile systems.

As routing becomes increasingly elaborate, we begin to see the border between *workflows* and *agents* grow clearer. Remember, we've defined 'autonomous agents' as AI systems that decide their own path (and the number of steps it will take) to accomplish a given task. A particularly sophisticated type of routing that leads to autonomous behavior is something we might call *evaluative routing*—where the model evaluates the output it produces and decides if it needs more work or is ready to be handed back to the user.

By making decisions based on its own qualitative evaluations, our agentic system can begin behaving like a totally unsupervised agent. Let's see it in practice.

#### Setup

First, let's import the necessary libraries and set up our connection to the OpenAI API:

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

def get_completion(prompt,
                   system_prompt="You are a helpful assistant.",
                   json_mode=False):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    if json_mode:
        response = client.chat.completions.create(
            model="gpt-6-luna",
            messages=messages,
            response_format={"type": "json_object"},
        )
    else:
        response = client.chat.completions.create(
            model="gpt-6-luna",
            messages=messages,
        )
    if response.choices[0].message.content is None:
        raise ValueError("The model returned no response.")
    return response.choices[0].message.content


Now, take a look at the following cell. In it, we use JSON mode to instruct the model to route a customer service request to either a human or a chatbot depending on a set of criteria. Try changing the customer support request in the `request` variable to a few different types of queries to see how the model behaves.

In [2]:
def route_customer(request):
    system_prompt = "You're a helpful assistant that outputs JSON. Do not include any markdown backticks in your response."
    prompt = f"""
    <instructions>Below you'll see a customer service request.
    Your job is to decide whether the request should be routed to a human or a chatbot.
    You should also provide a reason for your decision.
    The request is:
    {request}
    </instructions>
    <format>
    The JSON response should have two fields:
    - `decision`: "human" or "chatbot"
    - `reason`: a short explanation for your decision

    Route to a human if:
    - The request involves a complaint or dissatisfaction
    - The request requires account changes or financial transactions
    - The request mentions legal issues or threatens escalation
    
    Route to a chatbot if:
    - The request is for basic information or FAQs
    - The request is for simple status updates
    - The request is for general product information

    For instance:
    {{
        "decision": "human",
        "reason": "The request is a complex issue that requires a human touch."
    }}
    </format>
    """
    response = get_completion(prompt, system_prompt, json_mode=True)
    return response

request = "I need help with my account."
decision = route_customer(request)
json_decision = json.loads(decision)
decision = json_decision['decision']
reason = json_decision['reason']

print("Request:", request)
print("Decision: ", decision)
print("Reason: ", reason)

Request: I need help with my account.
Decision:  chatbot
Reason:  The request is general and doesn't specify an account change or other issue requiring a human; a chatbot can clarify what help is needed.


#### Checkpoint 1/3

Now, assuming you've got the `decision` and `reason` variables accessible from the last cell, let's write a little code that routes the LLM's decision to two different outcomes. We've defined two placeholder functions `handle_by_chatbot` and `handle_by_human`, which should be called depending on the content of the `decision` variable.

In the second half of the cell, write a control flow that calls `handle_by_chatbot` if `decision` is equal to `"chatbot"` and `handle_by_human` if `decision` is equal to `"human"`.

In [4]:
def handle_by_chatbot(request):
    print(f"Chatbot is handling: '{request}'")
    print(f"Reason: {reason}")
    print("Providing automated response...")
    print("Thank you for your inquiry. Here's the information you requested.")

def handle_by_human(request):
    print(f"Request escalated to human agent: '{request}'")
    print(f"Reason: {reason}")
    print("Sent to customer service human...")
    print("Your request has been assigned to a customer service representative who will contact you shortly.")

# Route the request based on the decision
print("\nRouting the request...")
## YOUR SOLUTION HERE ##
if decision == "human":
    handle_by_human(request)
elif decision == "chatbot":
    handle_by_chatbot(request)
else:
    handle_by_chatbot(request)



Routing the request...
Chatbot is handling: 'I need help with my account.'
Reason: The request is general and doesn't specify an account change or other issue requiring a human; a chatbot can clarify what help is needed.
Providing automated response...
Thank you for your inquiry. Here's the information you requested.


Ultimately, this form of routing is very simple: parse the JSON, send the results through a control flow, and execute the related code.

As we discussed earlier, *evaluative routing* unlocks more complex autonomous systems with less predictable sequences of actions.

In this type of routing, the model must decide whether or not it thinks a given sub-task is completed at a sufficiently high quality to proceed to the next step. If the output isn't found satisfactory, the model will be prompted to try again, perhaps including feedback that can help it improve.

We'll illustrate this idea through creating a recipe generator. At the end of the workflow, we'll add a prompt that checks if the recipe meets our set of criteria; if it doesn't, we'll send it back to the LLM for more edits.

Read through the following prompt and then we can begin to implement it.

In [5]:
recipe_criteria = """
Please evaluate the following recipe for {dish} based on these criteria:
1. Completeness: Does it have all necessary ingredients and steps?
2. Clarity: Are the instructions clear and easy to follow?
3. Culinary Accuracy: Are the cooking techniques appropriate?
4. Feasibility: Can a home cook reasonably make this dish?
5. Flavor complexity: Does it have a good balance of flavors?
6. Texture contrast: Does it have a good texture contrast?
7. Regional authenticity: Does it respect the dish's culinary tradition (if applicable)?
"""

def get_recipe_evaluation_prompt(recipe, dish):
    return f"""You are a professional chef evaluating recipe quality for {dish}.
    
    <criteria>
    {recipe_criteria}
    </criteria>
    
    <recipe>
    {recipe}
    </recipe>
    
    <instructions>
    Respond with a JSON object containing:
    1. "decision": Either "good" (meets standards) or "needs_work" (requires revision)
    2. "reason": Brief explanation of your decision
    3. "feedback": Specific suggestions for improvement if needed. 
    (If the recipe is good, 3. should be an empty string.)
    </instructions>
    
    <format>
    {{
      "decision": "good or needs_work",
      "reason": "your reasoning here",
      "feedback": "your specific feedback here"
    }}
    </format>

    <output_rules>
    Output ONLY the JSON, nothing else.
    Do not include any markdown backticks in your response.
    </output_rules>
    """


Each recipe will we judged against the prompt above. Recipes deemed `"good"` will be returned to the user. Recipes that still need work, however, will send back with the following prompt for revisions.

In [6]:
def get_recipe_revision_prompt(recipe, dish, feedback):
    return f"""You are a professional chef. Use the following feedback to improve the recipe for {dish}.
    Return the revised recipe in the same format as the original recipe. Output ONLY the recipe with no other commentary.
    
    <recipe>
    {recipe}
    </recipe>

    <criteria>
    {recipe_criteria}
    </criteria>

    <feedback>
    {feedback}
    </feedback>
    
    Your improved recipe:"""
    

#### Checkpoint 2/3
Implement an evaluative routing system using the prompts above.

The model's `decision` will be either `"good"` or `"needs_work"`. If `"good"`, we'll return the recipe. If `"needs_work"`, we'll send the recipe to the `get_recipe_revision_prompt` alongside its `"feedback"` and dish name. We'll send the result back to the original decision prompt to see if it passes.

We've obtained the `evaluation_response` for you at the start of the function. Your tasks are as follows:
- load the JSON in the evaluation and assign it to `evaluation_data`
- extract the `decision` from the evaluation and store it in `decision`
- if the decision is `good`, print the statement `"The recipe is good!"` and return the recipe
- if the decision is `needs_work`, do the following:
    - print `"The recipe needs work."`
    - create the revision prompt using the function defined above, the `recipe` and `dish` arguments, and the `feedback` portion of the evaluation data
    - get the `revised_recipe` from the LLM
    - return another call of the `route_recipe_request` function using the revised recipe and the dish 

For now, this process will continue indefinitely via recursion until the recipe is deemed `"good"` by the LLM.

At the bottom of the cell, we'll pass in an obviously poor spaghetti bolognese recipe and see how the model improves it.

In [ ]:
import json

def route_recipe_request(recipe, dish):
    evaluation_prompt = get_recipe_evaluation_prompt(recipe, dish)
    evaluation_response = get_completion(evaluation_prompt, json_mode=True)
    
    # Parse the JSON response
    if evaluation_response is None:
        raise ValueError("The model returned no response.")
    evaluation_data = json.loads(evaluation_response)
    decision = evaluation_data['decision']
    
    print(f"Evaluation decision: {decision}")
    
    if decision == "good":
        print(f"Reason: {evaluation_data['reason']}")
        return recipe
    else:
        ## YOUR SOLUTION HERE ##
        print(f"Reason: {evaluation_data['reason']}")
        print(f"Feedback: {evaluation_data['feedback']}")
        
        revision_prompt = get_recipe_revision_prompt(recipe, dish, evaluation_data['feedback'])
        revised_recipe = get_completion(revision_prompt)
        print(f"Revised recipe: {revised_recipe}")
        return route_recipe_request(revised_recipe, dish)
        

# Test the routing function
test_recipe = """
Quick Spaghetti
Ingredients:
- Pasta
- Sauce
Instructions:
1. Cook pasta
2. Add sauce
"""
test_dish = "spaghetti bolognese"
result = route_recipe_request(test_recipe, test_dish)

Evaluation decision: needs_work
Reason: This is too vague to make spaghetti bolognese: it lists no specific ingredients and gives no usable cooking details.
Feedback: List the ingredients for a bolognese sauce, such as ground meat, onion, carrot, celery, tomato, and seasonings. Explain how to prepare and simmer the sauce, specify the pasta and cooking time, and describe how to combine and serve them. For a more traditional approach, allow time for the sauce to simmer and develop flavor.
Revised recipe: Spaghetti Bolognese

Ingredients:
- 400 g spaghetti
- 300 g ground beef
- 150 g ground pork
- 1 small onion, finely diced
- 1 carrot, finely diced
- 1 celery stalk, finely diced
- 2 tbsp olive oil
- 2 tbsp tomato paste
- 400 g canned crushed tomatoes
- ½ cup dry white wine
- ½ cup whole milk
- 1 cup beef stock or water, as needed
- Salt and black pepper
- Grated Parmesan, for serving

Instructions:
1. Heat the olive oil in a large pot over medium heat. Add the onion, carrot, and celery a

#### Checkpoint 3/3

There's an obvious problem with the evaluative routing code we just wrote: it doesn't know when to stop! Imagine if for some reason the model consistently failed the evaluation no matter what edits it made. We'd bombard the server with requests, running up a hefty API bill in the process.

To finish up this evaluative routing in a responsible way, let's revise it to include a simple rate limiting system.

Add two named keywords to the main function here, `max_tries` and `current_try`. If the recipe needs work, first check if the current try is greater than or equal to the maximum number of tries. If it is, return the recipe. If it isn't, call the `get_recipe_revision_prompt` same as we did in the last checkpoint, obtain its result and assign it to `revised_recipe`, and recursively return a call to the `route_recipe_request` function. Pass `max_tries` and `current_try + 1` in the function call so it effectively increments.

In [ ]:
# Add two additional named arguments
def route_recipe_request(recipe, dish, max_tries=3, current_try=1): ## YOUR SOLUTION HERE ##
    evaluation_prompt = get_recipe_evaluation_prompt(recipe, dish)
    evaluation_response = get_completion(evaluation_prompt, json_mode=True)
    
    # Parse the JSON response
    if evaluation_response is None:
        raise ValueError("The model returned no response.")
    evaluation_data = json.loads(evaluation_response)
    decision = evaluation_data["decision"]
    
    print(f"Evaluation decision: {decision}")
    
    if decision == "good":
        print("Recipe meets quality standards! Returning to user.")
        print(f"Reason: {evaluation_data['reason']}")
        return recipe
    else:
        print(f"Recipe needs work. Sending to revision prompt.")
        print(f"Reason: {evaluation_data['reason']}")
        print(f"Feedback: {evaluation_data['feedback']}")
        
        # Check if we've reached the maximum number of tries
        ## YOUR SOLUTION HERE
        if current_try >= max_tries:
            print("Maximum number of tries reached. Returning the last revised recipe.")
            return recipe
        
        revision_prompt = get_recipe_revision_prompt(recipe, dish, evaluation_data['feedback'])
        revised_recipe = get_completion(revision_prompt)
        print(f"Revised recipe: {revised_recipe}")
        return route_recipe_request(revised_recipe, dish, max_tries, current_try + 1)
        

test_recipe = """
Cinnamon Bun Pancakes
Ingredients:
- Flour
- Milk
- Eggs
- Cinnamon sugar
Instructions:
1. Mix ingredients
2. Cook on pan
3. Top with cinnamon sugar
"""
test_dish = "fluffy breakfast pancakes with a cinnamon bun flavor"
result = route_recipe_request(test_recipe, test_dish, max_tries=3, current_try=1)

Evaluation decision: needs_work
Recipe needs work. Sending to revision prompt.
Reason: The recipe is too vague to reliably make fluffy pancakes and does not yet deliver a convincing cinnamon bun flavor.
Feedback: Provide ingredient quantities and add a leavener such as baking powder, plus salt and fat for flavor and tenderness. Separate wet and dry ingredients, mix just until combined, and give heat cues and cooking times. For cinnamon bun character, add a cinnamon-brown-sugar swirl or filling and finish with a simple icing or cream-cheese glaze rather than relying only on cinnamon sugar.
Revised recipe: Cinnamon Bun Pancakes

Ingredients:
- 1 ½ cups all-purpose flour
- 2 tablespoons brown sugar
- 2 teaspoons baking powder
- ½ teaspoon salt
- 1 teaspoon ground cinnamon
- 1 large egg
- 1 ¼ cups milk
- 3 tablespoons melted butter, divided
- 1 teaspoon vanilla extract
- ¼ cup packed brown sugar (for the swirl)
- 1 teaspoon ground cinnamon (for the swirl)
- 1 tablespoon all-purpose flour (

## Conclusion

Excellent! You've now got a fully-functional evaluative routing pattern that can be dropped into more elaborate workflows.

With just the four simple tools we've covered in this lesson, we can build powerful AI systems that can handle all kinds of tricky tasks. What remains for us to learn is how to put them all together in an effective whole.